# Kaggle Predict F1 Pit Stops: Challenger Models and Feature Importance

Neural-network challenger, CatBoost/RealMLP challenger experiments, and feature-importance analysis.


## CNN Baseline


## 1. Setup

In [ ]:
import gc
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.labelsize": 10, "legend.frameon": False})

RANDOM_STATE = 42
TARGET = "PitNextLap"
ID_COL = "id"

# CNNs are slower than LightGBM here. Start small, then scale if promising.
RUN_FAST = True
FAST_SAMPLE_SIZE = 120_000
N_SPLITS = 3 if RUN_FAST else 5
EPOCHS = 15 if RUN_FAST else 35
BATCH_SIZE = 2048

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

## 2. Load Data

In [ ]:
def find_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e5"),
        Path("/kaggle/input/playground-series-s6e5"),
        Path("../input/competitions/playground-series-s6e5"),
        Path("../input/playground-series-s6e5"),
        Path("data"),
        Path("../data"),
        Path("."),
    ]
    for path in candidates:
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError("Could not find train.csv and test.csv. Update DATA_DIR manually.")


def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="integer")
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif pd.api.types.is_object_dtype(dtype):
            nunique = out[col].nunique(dropna=False)
            if nunique / max(len(out), 1) < 0.5:
                out[col] = out[col].astype("category")
    return out


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = reduce_memory_usage(pd.read_csv(DATA_DIR / "train.csv"))
test = reduce_memory_usage(pd.read_csv(DATA_DIR / "test.csv"))
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if RUN_FAST and len(train) > FAST_SAMPLE_SIZE:
    train_eval = (
        train.groupby(TARGET, group_keys=False)
        .apply(lambda x: x.sample(frac=min(1.0, FAST_SAMPLE_SIZE / len(train)), random_state=RANDOM_STATE))
        .sample(frac=1.0, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )
else:
    train_eval = train.copy()

print(f"DATA_DIR: {DATA_DIR}")
print(f"train_eval: {train_eval.shape}")
print(f"test: {test.shape}")
print(f"target positive rate: {train_eval[TARGET].mean():.5f}")

## 3. Feature Engineering

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6

    if {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedRaceLaps"] = out["LapNumber"] / out["RaceProgress"].clip(lower=eps)
        out["EstimatedLapsRemaining"] = out["EstimatedRaceLaps"] - out["LapNumber"]
        out["LapNumber_x_RaceProgress"] = out["LapNumber"] * out["RaceProgress"]

    if {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreLife_to_LapNumber"] = out["TyreLife"] / out["LapNumber"].clip(lower=eps)

    if {"TyreLife", "EstimatedRaceLaps"}.issubset(out.columns):
        out["TyreLife_to_EstimatedRaceLaps"] = out["TyreLife"] / out["EstimatedRaceLaps"].clip(lower=eps)

    if {"LapTime (s)", "LapTime_Delta"}.issubset(out.columns):
        out["LapTime_plus_Delta"] = out["LapTime (s)"] + out["LapTime_Delta"]
        out["AbsLapTime_Delta"] = out["LapTime_Delta"].abs()

    if {"Position", "Position_Change"}.issubset(out.columns):
        out["PreviousPositionApprox"] = out["Position"] - out["Position_Change"]
        out["AbsPosition_Change"] = out["Position_Change"].abs()

    if "Compound" in out.columns:
        compound = out["Compound"].astype(str)
        out["IsSoft"] = (compound == "SOFT").astype("int8")
        out["IsMedium"] = (compound == "MEDIUM").astype("int8")
        out["IsHard"] = (compound == "HARD").astype("int8")
        out["IsWetOrIntermediate"] = compound.isin(["WET", "INTERMEDIATE"]).astype("int8")

    return reduce_memory_usage(out)


train_model = add_features(train_eval)
test_model = add_features(test)

feature_cols = [c for c in train_model.columns if c not in [TARGET, ID_COL]]
categorical_cols = train_model[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", categorical_cols)

## 4. Preprocessing Helpers

In [ ]:
def fit_transform_fold(X_train: pd.DataFrame, X_valid: pd.DataFrame, X_test: pd.DataFrame):
    numeric_imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    ordinal = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

    X_train_num = scaler.fit_transform(numeric_imputer.fit_transform(X_train[numeric_cols])).astype("float32")
    X_valid_num = scaler.transform(numeric_imputer.transform(X_valid[numeric_cols])).astype("float32")
    X_test_num = scaler.transform(numeric_imputer.transform(X_test[numeric_cols])).astype("float32")

    X_train_cat = ordinal.fit_transform(X_train[categorical_cols]).astype("int32") + 1
    X_valid_cat = ordinal.transform(X_valid[categorical_cols]).astype("int32") + 1
    X_test_cat = ordinal.transform(X_test[categorical_cols]).astype("int32") + 1

    cardinalities = [int(X_train_cat[:, i].max()) + 2 for i in range(X_train_cat.shape[1])]
    return X_train_num, X_valid_num, X_test_num, X_train_cat, X_valid_cat, X_test_cat, cardinalities


def evaluate_predictions(y_true, y_pred) -> dict:
    y_pred = np.clip(y_pred, 1e-6, 1 - 1e-6)
    return {
        "roc_auc": roc_auc_score(y_true, y_pred),
        "average_precision": average_precision_score(y_true, y_pred),
        "log_loss": log_loss(y_true, y_pred),
    }

## 5. CNN Model

In [ ]:
def build_cnn_model(n_numeric: int, cardinalities: list[int]) -> keras.Model:
    numeric_input = keras.Input(shape=(n_numeric,), name="numeric")
    categorical_input = keras.Input(shape=(len(cardinalities),), dtype="int32", name="categorical")

    embeddings = []
    for i, cardinality in enumerate(cardinalities):
        dim = min(16, max(4, int(np.sqrt(cardinality))))
        emb = layers.Embedding(input_dim=cardinality, output_dim=dim, name=f"emb_{i}")(categorical_input[:, i])
        embeddings.append(layers.Flatten()(emb))

    x = layers.Concatenate()(embeddings + [numeric_input])
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation="swish")(x)
    x = layers.Dropout(0.15)(x)
    x = layers.Reshape((128, 1))(x)
    x = layers.Conv1D(64, kernel_size=3, padding="same", activation="swish")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(32, kernel_size=3, padding="same", activation="swish")(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="swish")(x)
    x = layers.Dropout(0.20)(x)
    output = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs=[numeric_input, categorical_input], outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc"), keras.metrics.AUC(name="pr_auc", curve="PR")],
    )
    return model

## 6. Cross-Validation

In [ ]:
X = train_model[feature_cols].copy()
y = train_model[TARGET].astype("int8").copy()
X_test = test_model[feature_cols].copy()

class_values = np.array([0, 1])
class_weights = compute_class_weight(class_weight="balanced", classes=class_values, y=y)
class_weight = {int(cls): float(weight) for cls, weight in zip(class_values, class_weights)}
class_weight

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof = np.zeros(len(X), dtype=np.float32)
test_predictions = np.zeros(len(X_test), dtype=np.float32)
fold_rows = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    print(f"\n=== Fold {fold}/{N_SPLITS} ===")
    start = time.time()

    X_train_raw, X_valid_raw = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    X_train_num, X_valid_num, X_test_num, X_train_cat, X_valid_cat, X_test_cat, cardinalities = fit_transform_fold(
        X_train_raw, X_valid_raw, X_test
    )

    tf.keras.backend.clear_session()
    model = build_cnn_model(X_train_num.shape[1], cardinalities)
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", patience=2, factor=0.5, min_lr=1e-5),
    ]

    history = model.fit(
        {"numeric": X_train_num, "categorical": X_train_cat},
        y_train,
        validation_data=({"numeric": X_valid_num, "categorical": X_valid_cat}, y_valid),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=1,
    )

    valid_pred = model.predict({"numeric": X_valid_num, "categorical": X_valid_cat}, batch_size=BATCH_SIZE, verbose=0).ravel()
    test_pred = model.predict({"numeric": X_test_num, "categorical": X_test_cat}, batch_size=BATCH_SIZE, verbose=0).ravel()
    oof[valid_idx] = valid_pred
    test_predictions += test_pred / N_SPLITS

    scores = evaluate_predictions(y_valid, valid_pred)
    scores.update({"fold": fold, "epochs": len(history.history["loss"]), "fit_seconds": time.time() - start})
    fold_rows.append(scores)
    print(scores)

    del model, X_train_num, X_valid_num, X_test_num, X_train_cat, X_valid_cat, X_test_cat
    gc.collect()

overall = evaluate_predictions(y, oof)
overall

## 7. Results and Submission

In [ ]:
fold_results = pd.DataFrame(fold_rows)
display(fold_results)
print("OOF:", overall)

submission = sample_submission.copy()
submission[TARGET] = np.clip(test_predictions, 1e-6, 1 - 1e-6)
submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
fold_results.to_csv(OUTPUT_DIR / "cnn_baseline_fold_results.csv", index=False)

print("Saved:", submission_path)
submission.head()

## 8. Decision Rule

Treat this as a challenger baseline. Keep investing in neural models only if the CNN gets near the tuned LightGBM/ensemble validation range. If it trails clearly, tree models remain the best path and the CNN can still be useful as a low-weight ensemble member.

## CatBoost, RealMLP, and Feature Importance


## 1. Setup

In [ ]:
import gc
import importlib
import subprocess
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.labelsize": 10, "legend.frameon": False})

RANDOM_STATE = 42
TARGET = "PitNextLap"
ID_COL = "id"

RUN_FAST = True
FAST_SAMPLE_SIZE = 180_000
N_SPLITS = 5

# Set True only if your runtime allows internet/package installs.
INSTALL_REALMLP_IF_MISSING = False
RUN_REALMLP = True

## 2. Optional Dependencies

In [ ]:
try:
    from catboost import CatBoostClassifier, Pool
    CATBOOST_AVAILABLE = True
except Exception as exc:
    CATBOOST_AVAILABLE = False
    print("CatBoost unavailable:", exc)


def import_realmlp():
    try:
        from pytabkit import RealMLP_TD_Classifier
        return RealMLP_TD_Classifier, True
    except Exception as exc:
        print("RealMLP unavailable:", exc)
        if INSTALL_REALMLP_IF_MISSING:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pytabkit"])
            from pytabkit import RealMLP_TD_Classifier
            return RealMLP_TD_Classifier, True
        return None, False


RealMLP_TD_Classifier, REALMLP_AVAILABLE = import_realmlp()
print("CATBOOST_AVAILABLE:", CATBOOST_AVAILABLE)
print("REALMLP_AVAILABLE:", REALMLP_AVAILABLE)

## 3. Load Data

In [ ]:
def find_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e5"),
        Path("/kaggle/input/playground-series-s6e5"),
        Path("../input/competitions/playground-series-s6e5"),
        Path("../input/playground-series-s6e5"),
        Path("data"),
        Path("../data"),
        Path("."),
    ]
    for path in candidates:
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError("Could not find train.csv and test.csv. Update DATA_DIR manually.")


def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="integer")
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif pd.api.types.is_object_dtype(dtype):
            nunique = out[col].nunique(dropna=False)
            if nunique / max(len(out), 1) < 0.5:
                out[col] = out[col].astype("category")
    return out


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = reduce_memory_usage(pd.read_csv(DATA_DIR / "train.csv"))
test = reduce_memory_usage(pd.read_csv(DATA_DIR / "test.csv"))
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if RUN_FAST and len(train) > FAST_SAMPLE_SIZE:
    train_eval = (
        train.groupby(TARGET, group_keys=False)
        .apply(lambda x: x.sample(frac=min(1.0, FAST_SAMPLE_SIZE / len(train)), random_state=RANDOM_STATE))
        .sample(frac=1.0, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )
else:
    train_eval = train.copy()

print(f"DATA_DIR: {DATA_DIR}")
print(f"train_eval: {train_eval.shape}")
print(f"test: {test.shape}")
print(f"target positive rate: {train_eval[TARGET].mean():.5f}")

## 4. Feature Engineering

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6

    if {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedRaceLaps"] = out["LapNumber"] / out["RaceProgress"].clip(lower=eps)
        out["EstimatedLapsRemaining"] = out["EstimatedRaceLaps"] - out["LapNumber"]
        out["LapNumber_x_RaceProgress"] = out["LapNumber"] * out["RaceProgress"]

    if {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreLife_to_LapNumber"] = out["TyreLife"] / out["LapNumber"].clip(lower=eps)

    if {"TyreLife", "EstimatedRaceLaps"}.issubset(out.columns):
        out["TyreLife_to_EstimatedRaceLaps"] = out["TyreLife"] / out["EstimatedRaceLaps"].clip(lower=eps)

    if {"LapTime (s)", "LapTime_Delta"}.issubset(out.columns):
        out["LapTime_plus_Delta"] = out["LapTime (s)"] + out["LapTime_Delta"]
        out["AbsLapTime_Delta"] = out["LapTime_Delta"].abs()

    if {"Position", "Position_Change"}.issubset(out.columns):
        out["PreviousPositionApprox"] = out["Position"] - out["Position_Change"]
        out["AbsPosition_Change"] = out["Position_Change"].abs()

    if "Compound" in out.columns:
        compound = out["Compound"].astype(str)
        out["IsSoft"] = (compound == "SOFT").astype("int8")
        out["IsMedium"] = (compound == "MEDIUM").astype("int8")
        out["IsHard"] = (compound == "HARD").astype("int8")
        out["IsWetOrIntermediate"] = compound.isin(["WET", "INTERMEDIATE"]).astype("int8")

    # Feature validation preferred dropping Driver by a tiny margin.
    if "Driver" in out.columns:
        out = out.drop(columns=["Driver"])

    return reduce_memory_usage(out)


train_model = add_features(train_eval)
test_model = add_features(test)

feature_cols = [c for c in train_model.columns if c not in [TARGET, ID_COL]]
categorical_cols = train_model[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

X = train_model[feature_cols].copy()
y = train_model[TARGET].astype("int8").copy()
X_test = test_model[feature_cols].copy()

print("Features:", len(feature_cols))
print("Categorical:", categorical_cols)
print("Numeric:", len(numeric_cols))

## 5. Shared Metrics and Preprocessing

In [ ]:
def evaluate_predictions(y_true, y_pred) -> dict:
    y_pred = np.clip(y_pred, 1e-6, 1 - 1e-6)
    return {
        "roc_auc": roc_auc_score(y_true, y_pred),
        "average_precision": average_precision_score(y_true, y_pred),
        "log_loss": log_loss(y_true, y_pred),
    }


def make_lgbm_preprocessor(df: pd.DataFrame) -> ColumnTransformer:
    num_cols = [c for c in df.columns if c in numeric_cols]
    cat_cols = [c for c in df.columns if c in categorical_cols]
    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ], remainder="drop")


def prepare_catboost_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in categorical_cols:
        out[col] = out[col].astype(str).fillna("missing")
    return out


cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

## 6. Model Configurations

In [ ]:
lgbm_params = {
    "objective": "binary",
    "n_estimators": 900 if RUN_FAST else 1800,
    "learning_rate": 0.02,
    "num_leaves": 127,
    "min_child_samples": 150,
    "subsample": 0.95,
    "colsample_bytree": 0.9,
    "reg_alpha": 1.0,
    "reg_lambda": 10.0,
    "max_bin": 255,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": -1,
}

catboost_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": 1200 if RUN_FAST else 2500,
    "learning_rate": 0.035,
    "depth": 7,
    "l2_leaf_reg": 6.0,
    "random_seed": RANDOM_STATE,
    "allow_writing_files": False,
    "verbose": False,
    "thread_count": -1,
}

realmlp_params = {
    "random_state": RANDOM_STATE,
    "n_cv": 1,
    "val_fraction": 0.2,
    "n_epochs": 128 if RUN_FAST else 256,
    "batch_size": 1024,
    "verbosity": 0,
}

## 7. LightGBM Importance Baseline

In [ ]:
def cross_validate_lgbm():
    oof = np.zeros(len(X), dtype=np.float32)
    rows = []
    importance_rows = []
    start = time.time()

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        X_train_raw, X_valid_raw = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        preprocessor = make_lgbm_preprocessor(X_train_raw)
        X_train = preprocessor.fit_transform(X_train_raw)
        X_valid = preprocessor.transform(X_valid_raw)
        feature_names = preprocessor.get_feature_names_out()
        feature_names = [name.split("__", 1)[-1] for name in feature_names]

        model = LGBMClassifier(**lgbm_params)
        model.fit(X_train, y_train)
        pred = model.predict_proba(X_valid)[:, 1]
        oof[valid_idx] = pred

        scores = evaluate_predictions(y_valid, pred)
        scores.update({"model": "lightgbm", "fold": fold})
        rows.append(scores)

        for feature, gain, split in zip(feature_names, model.booster_.feature_importance("gain"), model.booster_.feature_importance("split")):
            importance_rows.append({"model": "lightgbm", "fold": fold, "feature": feature, "importance_gain": gain, "importance_split": split})

        del preprocessor, X_train, X_valid, model
        gc.collect()

    overall = evaluate_predictions(y, oof)
    overall.update({"model": "lightgbm", "fold": "oof", "fit_seconds": time.time() - start})
    rows.append(overall)
    return oof, rows, pd.DataFrame(importance_rows)


lgbm_oof, lgbm_rows, lgbm_importance = cross_validate_lgbm()
pd.DataFrame(lgbm_rows).tail(1)

## 8. CatBoost Challenger

In [ ]:
def cross_validate_catboost():
    if not CATBOOST_AVAILABLE:
        return None, [], pd.DataFrame()

    X_cb = prepare_catboost_frame(X)
    cat_features = [X_cb.columns.get_loc(col) for col in categorical_cols]
    oof = np.zeros(len(X_cb), dtype=np.float32)
    rows = []
    importance_rows = []
    start = time.time()

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_cb, y), start=1):
        X_train, X_valid = X_cb.iloc[train_idx], X_cb.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        train_pool = Pool(X_train, y_train, cat_features=cat_features)
        valid_pool = Pool(X_valid, y_valid, cat_features=cat_features)

        model = CatBoostClassifier(**catboost_params)
        model.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=120)
        pred = model.predict_proba(valid_pool)[:, 1]
        oof[valid_idx] = pred

        scores = evaluate_predictions(y_valid, pred)
        scores.update({"model": "catboost", "fold": fold, "best_iteration": model.get_best_iteration()})
        rows.append(scores)

        importances = model.get_feature_importance(train_pool, type="PredictionValuesChange")
        for feature, importance in zip(X_cb.columns, importances):
            importance_rows.append({"model": "catboost", "fold": fold, "feature": feature, "importance": importance})

        del train_pool, valid_pool, model
        gc.collect()

    overall = evaluate_predictions(y, oof)
    overall.update({"model": "catboost", "fold": "oof", "fit_seconds": time.time() - start})
    rows.append(overall)
    return oof, rows, pd.DataFrame(importance_rows)


catboost_oof, catboost_rows, catboost_importance = cross_validate_catboost()
pd.DataFrame(catboost_rows).tail(1) if catboost_rows else "CatBoost unavailable"

## 9. Optional RealMLP Challenger

In [ ]:
def cross_validate_realmlp():
    if not (RUN_REALMLP and REALMLP_AVAILABLE):
        return None, []

    X_real = X.copy()
    for col in categorical_cols:
        X_real[col] = X_real[col].astype("category")

    oof = np.zeros(len(X_real), dtype=np.float32)
    rows = []
    start = time.time()

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_real, y), start=1):
        X_train, X_valid = X_real.iloc[train_idx], X_real.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = RealMLP_TD_Classifier(**realmlp_params)
        model.fit(X_train, y_train)
        pred = model.predict_proba(X_valid)[:, 1]
        oof[valid_idx] = pred

        scores = evaluate_predictions(y_valid, pred)
        scores.update({"model": "realmlp", "fold": fold})
        rows.append(scores)

        del model
        gc.collect()

    overall = evaluate_predictions(y, oof)
    overall.update({"model": "realmlp", "fold": "oof", "fit_seconds": time.time() - start})
    rows.append(overall)
    return oof, rows


realmlp_oof, realmlp_rows = cross_validate_realmlp()
pd.DataFrame(realmlp_rows).tail(1) if realmlp_rows else "RealMLP skipped or unavailable"

## 10. Compare Models

In [ ]:
all_rows = lgbm_rows + catboost_rows + realmlp_rows
results = pd.DataFrame(all_rows)
summary = results[results["fold"].eq("oof")].sort_values("roc_auc", ascending=False).reset_index(drop=True)
summary

## 11. Feature Importance

In [ ]:
lgbm_imp = (
    lgbm_importance.groupby("feature", as_index=False)
    .agg(importance_gain=("importance_gain", "mean"), importance_split=("importance_split", "mean"))
    .sort_values("importance_gain", ascending=False)
)

display(lgbm_imp.head(25))

fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(data=lgbm_imp.head(25), y="feature", x="importance_gain", color=sns.color_palette("viridis", 8)[4], ax=ax)
ax.set_title("LightGBM feature importance, mean gain across folds")
ax.set_xlabel("Mean gain")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
if not catboost_importance.empty:
    cat_imp = (
        catboost_importance.groupby("feature", as_index=False)
        .agg(importance=("importance", "mean"))
        .sort_values("importance", ascending=False)
    )
    display(cat_imp.head(25))

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.barplot(data=cat_imp.head(25), y="feature", x="importance", color=sns.color_palette("viridis", 8)[5], ax=ax)
    ax.set_title("CatBoost feature importance, mean across folds")
    ax.set_xlabel("PredictionValuesChange")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()
else:
    print("CatBoost importance unavailable.")

## 12. Train Best Available Challenger and Submit

In [ ]:
best_model_name = summary.iloc[0]["model"]
print("Best OOF model:", best_model_name)

if best_model_name == "catboost" and CATBOOST_AVAILABLE:
    X_cb = prepare_catboost_frame(X)
    X_test_cb = prepare_catboost_frame(X_test)
    cat_features = [X_cb.columns.get_loc(col) for col in categorical_cols]
    train_pool = Pool(X_cb, y, cat_features=cat_features)
    test_pool = Pool(X_test_cb, cat_features=cat_features)
    final_model = CatBoostClassifier(**catboost_params)
    final_model.fit(train_pool)
    test_pred = final_model.predict_proba(test_pool)[:, 1]
elif best_model_name == "realmlp" and REALMLP_AVAILABLE:
    X_real = X.copy()
    X_test_real = X_test.copy()
    for col in categorical_cols:
        X_real[col] = X_real[col].astype("category")
        X_test_real[col] = X_test_real[col].astype("category")
    final_model = RealMLP_TD_Classifier(**realmlp_params)
    final_model.fit(X_real, y)
    test_pred = final_model.predict_proba(X_test_real)[:, 1]
else:
    preprocessor = make_lgbm_preprocessor(X)
    X_train_processed = preprocessor.fit_transform(X)
    X_test_processed = preprocessor.transform(X_test)
    final_model = LGBMClassifier(**lgbm_params)
    final_model.fit(X_train_processed, y)
    test_pred = final_model.predict_proba(X_test_processed)[:, 1]

submission = sample_submission.copy()
submission[TARGET] = np.clip(test_pred, 1e-6, 1 - 1e-6)
submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

summary.to_csv(OUTPUT_DIR / "catboost_realmlp_model_summary.csv", index=False)
lgbm_imp.to_csv(OUTPUT_DIR / "lightgbm_feature_importance.csv", index=False)
if not catboost_importance.empty:
    cat_imp.to_csv(OUTPUT_DIR / "catboost_feature_importance.csv", index=False)

print("Saved:", submission_path)
submission.head()

## 13. Decision Rules

- Keep CatBoost if it beats tuned LightGBM or adds diverse predictions for an ensemble.
- Keep RealMLP only if it runs reliably and gets close to the tree models.
- Use feature importance to guide feature pruning, not as proof of causality.
- If LightGBM and CatBoost importance agree, prioritize those features for slice/error analysis.